In [3]:
import numpy as np

def parse_outcar(filename, n_atoms=47):
    positions = []
    forces = []
    energies = []

    with open(filename, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i]

        # Look for start of POSITION and FORCE block
        if 'POSITION' in line and 'TOTAL-FORCE' in line:
            i += 2  # Skip dashed line
            pos_block = []
            frc_block = []

            for _ in range(n_atoms):
                parts = lines[i].strip().split()
                pos = list(map(float, parts[:3]))
                frc = list(map(float, parts[3:6]))
                pos_block.append(pos)
                frc_block.append(frc)
                i += 1

            positions.append(pos_block)
            forces.append(frc_block)

            # Now look ahead for the closest 'energy(sigma->0)' after this block
            while i < len(lines):
                if 'energy(sigma->0) =' in lines[i]:
                    energy = float(lines[i].strip().split('=')[-1])
                    energies.append(energy)
                    break
                i += 1
        else:
            i += 1

    # Convert lists to NumPy arrays
    positions = np.array(positions)
    forces = np.array(forces)
    energies = np.array(energies)

    return positions, forces, energies

def save_to_txt(positions, forces, energies, filename='vasp_trajectory.txt'):
    n_steps = positions.shape[0]
    n_atoms = positions.shape[1]

    with open(filename, 'w') as f:
        for step in range(n_steps):
            f.write(f'  point= {step:6d}  {energies[step]}\n')
            for atom in range(n_atoms):
                pos = positions[step][atom]
                frc = forces[step][atom]
                f.write(f'  {pos[0]:10.5f} {pos[1]:10.5f} {pos[2]:10.5f}  '
                        f'{frc[0]:10.6f} {frc[1]:10.6f} {frc[2]:10.6f}\n')

# Run it all
positions, forces, energies = parse_outcar('OUTCAR')
save_to_txt(positions, forces, energies, filename='vasp_trajectory.txt')

In [2]:
import numpy as np

def load_trajectory(filename, n_atoms=47):
    positions = []
    forces = []
    energies = []

    with open(filename, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if lines[i].startswith('  point='):
            # Extract energy
            energy = float(lines[i].strip().split()[-1])
            i += 1
            pos_block = []
            frc_block = []
            for _ in range(n_atoms):
                parts = list(map(float, lines[i].strip().split()))
                pos_block.append(parts[:3])
                frc_block.append(parts[3:])
                i += 1
            positions.append(pos_block)
            forces.append(frc_block)
            energies.append(energy)
        else:
            i += 1

    return np.array(positions), np.array(forces), np.array(energies)

def filter_unique_geometries(positions, forces, energies, tol=0.1, z_gap_cutoff=3.0):
    unique_pos = []
    unique_frc = []
    unique_en = []

    for i in range(len(positions)):
        current = positions[i]

        # --- Z gap criterion: reject if N atoms are too far from surface ---
        z45 = current[44][2]
        z46 = current[45][2]
        z47 = current[46][2]
        max_n_z = max(z46, z47)
        if (max_n_z - z45) > z_gap_cutoff:
            continue  # skip this structure

        # --- Duplicate check based on position difference ---
        is_duplicate = False
        for j in range(len(unique_pos)):
            diff = np.sum(np.abs(current - unique_pos[j]))
            if diff < tol:
                is_duplicate = True
                break

        if not is_duplicate:
            if energies[i] < -225.30239:
                unique_pos.append(current)
                unique_frc.append(forces[i])
                unique_en.append(energies[i])

    return np.array(unique_pos), np.array(unique_frc), np.array(unique_en)

def write_reann_format(positions, forces, energies, filename='reann_input.txt'):
    lattice = [
        [8.3643661146556720, 0.0000000000000000, 0.0000000000000000],
        [4.1821830573278360, 7.2437535418455532, 0.0000000000000000],
        [0.0000000000000000, 0.0000000000000000, 29.1059684456587782]
    ]
    
    with open(filename, 'w') as f:
        for idx in range(len(positions)):
            f.write(f'point= {idx + 1}\n')

            # Lattice
            for vec in lattice:
                f.write(f'  {vec[0]:20.16f} {vec[1]:20.16f} {vec[2]:20.16f}\n')

            # PBC
            f.write('pbc 1 1 1\n')

            # Atoms: first 45 Pd, then 2 N
            for atom_idx in range(47):
                pos = positions[idx][atom_idx]
                frc = forces[idx][atom_idx]
                if atom_idx < 45:
                    element = 'Pd'
                    mass = 106.420
                else:
                    element = 'N'
                    mass = 14.001
                f.write(f'{element:<2} {mass:7.3f} {pos[0]:10.6f} {pos[1]:10.6f} {pos[2]:10.6f} {frc[0]:10.6f} {frc[1]:10.6f} {frc[2]:10.6f}\n')

            # abprop line
            f.write(f'abprop: {energies[idx]:.15f}\n')

def random_split_and_save(positions, forces, energies, train_ratio=0.9, seed=42):
    # Set seed for reproducibility
    np.random.seed(seed)

    # Shuffle indices
    n_total = len(positions)
    indices = np.random.permutation(n_total)

    # Split indices into train and test
    n_train = int(n_total * train_ratio)
    train_indices = indices[:n_train]
    test_indices = indices[n_train:]

    # Split data accordingly
    train_pos, train_frc, train_en = positions[train_indices], forces[train_indices], energies[train_indices]
    test_pos, test_frc, test_en = positions[test_indices], forces[test_indices], energies[test_indices]

    # Save the split data
    write_reann_format(train_pos, train_frc, train_en, filename='train.txt')
    write_reann_format(test_pos, test_frc, test_en, filename='test.txt')

    print(f"Train set: {len(train_pos)} structures, Test set: {len(test_pos)} structures.")

# Run the full process
positions, forces, energies = load_trajectory('vasp_trajectory.txt')
unique_pos, unique_frc, unique_en = filter_unique_geometries(positions, forces, energies, tol=0.3)
random_split_and_save(unique_pos, unique_frc, unique_en, train_ratio=0.9)

print(f"Filtered from {len(positions)} to {len(unique_pos)} unique structures.")

Train set: 30 structures, Test set: 4 structures.
Filtered from 352 to 34 unique structures.


In [ ]:
import numpy as np

def load_trajectory(filename, n_atoms=47):
    positions = []
    forces = []
    energies = []

    with open(filename, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if lines[i].startswith('  point='):
            # Extract energy
            energy = float(lines[i].strip().split()[-1])
            i += 1
            pos_block = []
            frc_block = []
            for _ in range(n_atoms):
                parts = list(map(float, lines[i].strip().split()))
                pos_block.append(parts[:3])
                frc_block.append(parts[3:])
                i += 1
            positions.append(pos_block)
            forces.append(frc_block)
            energies.append(energy)
        else:
            i += 1

    return np.array(positions), np.array(forces), np.array(energies)

def filter_unique_geometries(positions, forces, energies, tol=0.1, z_gap_cutoff=3.0):
    unique_pos = []
    unique_frc = []
    unique_en = []

    for i in range(len(positions)):
        current = positions[i]

        # --- Z gap criterion: reject if N atoms are too far from surface ---
        z45 = current[44][2]
        z46 = current[45][2]
        z47 = current[46][2]
        max_n_z = max(z46, z47)
        if (max_n_z - z45) > z_gap_cutoff:
            continue  # skip this structure

        # --- Duplicate check based on position difference ---
        is_duplicate = False
        for j in range(len(unique_pos)):
            diff = np.sum(np.abs(current - unique_pos[j]))
            if diff < tol:
                is_duplicate = True
                break

        if not is_duplicate:
            unique_pos.append(current)
            unique_frc.append(forces[i])
            unique_en.append(energies[i])

    return np.array(unique_pos), np.array(unique_frc), np.array(unique_en)

def write_reann_format(positions, forces, energies, filename='reann_input.txt'):
    lattice = [
        [8.3643661146556720, 0.0000000000000000, 0.0000000000000000],
        [4.1821830573278360, 7.2437535418455532, 0.0000000000000000],
        [0.0000000000000000, 0.0000000000000000, 29.1059684456587782]
    ]
    
    with open(filename, 'w') as f:
        for idx in range(len(positions)):
            f.write(f'point= {idx + 1}\n')

            # Lattice
            for vec in lattice:
                f.write(f'  {vec[0]:20.16f} {vec[1]:20.16f} {vec[2]:20.16f}\n')

            # PBC
            f.write('pbc 1 1 1\n')

            # Atoms: first 45 Pd, then 2 N
            for atom_idx in range(47):
                pos = positions[idx][atom_idx]
                frc = forces[idx][atom_idx]
                if atom_idx < 45:
                    element = 'Pd'
                    mass = 106.420
                else:
                    element = 'N'
                    mass = 14.001
                f.write(f'{element:<2} {mass:7.3f} {pos[0]:10.6f} {pos[1]:10.6f} {pos[2]:10.6f} {frc[0]:10.6f} {frc[1]:10.6f} {frc[2]:10.6f}\n')

            # abprop line
            f.write(f'abprop: {energies[idx]:.15f}\n')

# Run the full process
positions, forces, energies = load_trajectory('vasp_trajectory.txt')
unique_pos, unique_frc, unique_en = filter_unique_geometries(positions, forces, energies, tol=0.3)
write_reann_format(unique_pos, unique_frc, unique_en, filename='reann_input.txt')


In [ ]:
def split_and_save(positions, forces, energies, train_ratio=0.9, seed=42):
    # Shuffle the indices
    np.random.seed(seed)
    indices = np.random.permutation(len(positions))

    # Split into train and test indices
    n_train = int(len(positions) * train_ratio)
    train_indices = indices[:n_train]
    test_indices = indices[n_train:]

    # Prepare train and test sets
    train_pos = positions[train_indices]
    train_frc = forces[train_indices]
    train_en = energies[train_indices]

    test_pos = positions[test_indices]
    test_frc = forces[test_indices]
    test_en = energies[test_indices]

    # Save to files
    save_filtered('train.txt', train_pos, train_frc, train_en)
    save_filtered('test.txt', test_pos, test_frc, test_en)

    print(f"Saved {len(train_pos)} structures to 'train.txt' and {len(test_pos)} to 'test.txt'")


positions, forces, energies = load_trajectory('vasp_trajectory.txt')
unique_pos, unique_frc, unique_en = filter_unique_geometries(positions, forces, energies, tol=0.3)
split_and_save(unique_pos, unique_frc, unique_en, train_ratio=0.9)

print(f"Filtered from {len(positions)} to {len(unique_pos)} unique structures.")